### Top 10 most in-demand skills

Insight: Which technologies will dominate market trends?

In [0]:
%%sql
SELECT 
    skill,
    job_count
FROM hive_metastore.gold_portfolio.gold_agg_top_skills
ORDER BY job_count DESC
LIMIT 10

### Average salary by seniority level
Insight: how much each level earns

In [0]:
%sql
SELECT 
    job_family,
    ROUND(AVG(avg_salary_min), 2) AS avg_salary,
    SUM(job_count) AS total_jobs
FROM hive_metastore.gold_portfolio.gold_agg_salary
GROUP BY job_family
ORDER BY avg_salary DESC

### Salary by area (job_family)
Insight: which area pays the most

In [0]:
%sql
SELECT 
    job_family,
    ROUND(AVG(avg_salary_min), 2) AS avg_salary,
    SUM(job_count) AS total_jobs
FROM hive_metastore.gold_portfolio.gold_agg_salary
GROUP BY job_family
ORDER BY avg_salary DESC

### Job openings by location
Insight: geographic concentration of job openings

In [0]:
%sql
SELECT 
    location_normalized,
    job_count
FROM hive_metastore.gold_portfolio.gold_agg_location
ORDER BY job_count DESC

### Skills by area
Insight: stack of each area

In [0]:
%sql
SELECT 
    f.job_family,
    s.skill,
    COUNT(*) AS total
FROM hive_metastore.gold_portfolio.gold_fact_job_skills fs
JOIN hive_metastore.gold_portfolio.gold_fact_jobs f 
    ON fs.job_id = f.job_id
JOIN hive_metastore.gold_portfolio.gold_dim_skills s 
    ON fs.skill_id = s.skill_id
GROUP BY f.job_family, s.skill
ORDER BY f.job_family, total DESC

### Top skills by area (best version)
insight: top 5 skills por área

In [0]:
%sql
WITH ranked AS (
    SELECT 
        f.job_family,
        s.skill,
        COUNT(*) AS total,
        ROW_NUMBER() OVER (
            PARTITION BY f.job_family 
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM hive_metastore.gold_portfolio.gold_fact_job_skills fs
    JOIN hive_metastore.gold_portfolio.gold_fact_jobs f 
        ON fs.job_id = f.job_id
    JOIN hive_metastore.gold_portfolio.gold_dim_skills s 
        ON fs.skill_id = s.skill_id
    GROUP BY f.job_family, s.skill
)

SELECT *
FROM ranked
WHERE rn <= 5

### Data quality (excellent for portfolio)
insight: Insight: Dataset quality

In [0]:
%sql
SELECT 
    COUNT(*) AS total_jobs,
    SUM(CASE WHEN salary_min IS NULL THEN 1 ELSE 0 END) AS missing_salary,
    SUM(CASE WHEN location_normalized IS NULL THEN 1 ELSE 0 END) AS missing_location
FROM hive_metastore.gold_portfolio.gold_fact_jobs

### Salary distribution
Insight: distribuição real (histograma depois)

In [0]:
%sql
SELECT 
    salary_min,
    COUNT(*) AS freq
FROM hive_metastore.gold_portfolio.gold_fact_jobs
WHERE salary_min IS NOT NULL
GROUP BY salary_min
ORDER BY salary_min

### Companies with the most job openings
Insight: who hires the most

In [0]:
%sql
SELECT 
    company,
    COUNT(*) AS total_jobs
FROM hive_metastore.gold_portfolio.gold_fact_jobs
GROUP BY company
ORDER BY total_jobs DESC
LIMIT 10

### Senioridade por área
Insight: Distribution of levels by area

In [0]:
%sql
SELECT 
    job_family,
    seniority,
    COUNT(*) AS total
FROM hive_metastore.gold_portfolio.gold_fact_jobs
GROUP BY job_family, seniority
ORDER BY job_family, total DESC